# 🐼 Panda AI — One-Click Deploy on Colab

Deploy the full Panda AI gateway (OpenAI-compatible API + Dashboard) in one click.

**Usage:** Runtime → Run all (Ctrl+F9)

After deploy:
1. Open Dashboard → paste your API token
2. Import cookies from your ChatGPT/Claude session
3. Use the API as an OpenAI-compatible endpoint

> ⏱️ ~3 min first run. Sessions last ~12h on free Colab.

## 1️⃣ Install Dependencies

In [ ]:
# ── Node.js 20 (Next.js requires 18+) ────────────────────────────────
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1

# ── Python deps + Chromium ────────────────────────────────────────────
!pip install -q -r requirements.txt --root-user-action=ignore 2>&1 | grep -v WARNING | tail -2
!patchright install chromium 2>&1 | tail -1

# ── Cloudflared ───────────────────────────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!mv /tmp/cloudflared /usr/local/bin/cloudflared 2>/dev/null || cp /tmp/cloudflared /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, sys
node_v = subprocess.check_output(['node', '--version']).decode().strip()
print(f'✅ Python {sys.version.split()[0]} | Node {node_v} | Chromium + cloudflared ready')

## 2️⃣ Clone & Configure

In [ ]:
import os, secrets

os.chdir('/')
!rm -rf /content/Panda-Ai
!git clone -q https://github.com/ferelking242/Panda-Ai.git /content/Panda-Ai
os.chdir('/content/Panda-Ai')

# Generate token + .env
api_token = 'pnd_' + secrets.token_hex(16)
with open('.env', 'w') as f:
    f.write(f'PROVIDER=chatgpt\nHEADLESS=true\nAPI_HOST=0.0.0.0\nAPI_PORT=8000\nAPI_TOKEN={api_token}\nPOOL_SIZE=1\nRESPONSE_TIMEOUT=120000\nLOG_LEVEL=INFO\n')

print(f'✅ Repo cloned | Provider: chatgpt')
print(f'🔑 Token: {api_token}')

## 3️⃣ Build Dashboard

In [ ]:
os.chdir('/content/Panda-Ai/dashboard')
!npm install --no-audit --no-fund --registry=https://registry.npmjs.org/ --silent 2>&1 | tail -1
!npm run build 2>&1 | tail -3
os.chdir('/content/Panda-Ai')
print('✅ Dashboard built')

## 4️⃣ Start Backend + Dashboard

In [ ]:
import subprocess, time, os, sys

!pkill -f 'uvicorn.*src.api.server' 2>/dev/null || true
!pkill -f 'node.*server.js' 2>/dev/null || true
time.sleep(1)

# Pass API_TOKEN as env var (belt-and-suspenders with .env + load_dotenv)
server_env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'API_TOKEN': api_token}

api_proc = subprocess.Popen(
    [sys.executable, '-m', 'src.api.server'],
    cwd='/content/Panda-Ai',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env=server_env
)
print(f'🚀 API server starting (PID {api_proc.pid})...')

dash_proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='/content/Panda-Ai/dashboard',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**server_env, 'PORT': '5000', 'API_ORIGIN': 'http://127.0.0.1:8000', 'NODE_ENV': 'production'}
)
print(f'📊 Dashboard starting (PID {dash_proc.pid})...')

import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3)
        print(f'✅ API healthy after {(i+1)*2}s')
        break
    except Exception:
        if i == 29:
            print('⚠️ API not yet healthy')

## 5️⃣ Expose Public URLs

In [ ]:
import re, threading

urls = {}

def start_tunnel(port, name):
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            urls[name] = match.group(0)
            print(f'  🔗 {name}: {match.group(0)}')
            break

t1 = threading.Thread(target=start_tunnel, args=(8000, 'API'))
t2 = threading.Thread(target=start_tunnel, args=(5000, 'Dashboard'))
t1.start()
t2.start()

for _ in range(30):
    time.sleep(1)
    if len(urls) >= 2:
        break

print()
print('═' * 55)
print('  🐼 PANDA AI — DEPLOYED')
print('═' * 55)
if 'API' in urls:
    print(f'  🤖 API:     {urls["API"]}/v1')
if 'Dashboard' in urls:
    print(f'  📊 Dashboard: {urls["Dashboard"]}')
print('═' * 55)
print(f'  🔑 Token: {api_token}')
print('═' * 55)

## 📋 Quick Test

In [ ]:
import urllib.request, json

base = 'http://127.0.0.1:8000'

# 1. Health check (no auth)
health = json.loads(urllib.request.urlopen(f'{base}/healthz').read())
print(f'✅ Health: {health}')

# 2. Models (with token)
try:
    req = urllib.request.Request(f'{base}/v1/models', headers={'Authorization': f'Bearer {api_token}'})
    models = json.loads(urllib.request.urlopen(req).read())
    print(f'✅ Models: {[m["id"] for m in models["data"][:5]]}')
except urllib.error.HTTPError as e:
    print(f'⚠️ /v1/models returned {e.code}')
    # Try reading bootstrap token as fallback
    try:
        with open('/content/Panda-Ai/.panda_bootstrap_token') as f:
            boot = f.read().strip()
        print(f'   Bootstrap token found: {boot[:12]}...')
        req2 = urllib.request.Request(f'{base}/v1/models', headers={'Authorization': f'Bearer {boot}'})
        models = json.loads(urllib.request.urlopen(req2).read())
        print(f'✅ Models (via bootstrap): {[m["id"] for m in models["data"][:5]]}')
        print(f'   → Use this token instead: {boot}')
    except Exception:
        print('   No bootstrap token either. Check server logs.')

# 3. Dashboard config (no auth)
try:
    config = json.loads(urllib.request.urlopen(f'{base}/api/dashboard/config').read())
    print(f'✅ Dashboard config: provider={config["provider"]}, headless={config["headless"]}')
except Exception as e:
    print(f'⚠️ Dashboard config: {e}')